In [45]:
from pathlib import Path
import pandas as pd
import re   # <--- IMPORTANTE


In [58]:
 
BASE = Path("/mnt/c/Data")

for p in BASE.iterdir():
    print("-", p.name)
# pon aquí exactamente lo que te salió en el print
nombres = [
    "DB1INDUSTRIALFARM1.xlsx",
    "DB2INDUSTRIALFARM2C1.xlsx",
    "DB3INDUSTRIALFARM2C2.xlsx",
    "DB4ACADEMICPONDS1.xlsx",
    "DB5ACADEMICPONDS2.xlsx",
]
dfs = []
for n in nombres:
    f = BASE / n
    print("Leyendo:", f)
    df = pd.read_excel(f)   # no hace falta engine="openpyxl"
    df["source_file"] = f.name
    dfs.append(df)

raw = pd.concat(dfs, ignore_index=True)
# Normalizamos nombres de columnas
raw.columns = [c.strip().lower().replace(" ", "_") for c in raw.columns]

rename = {}
for c in raw.columns:
    if "length" in c:        rename[c] = "longitud_cm"
    if "weight" in c:        rename[c] = "peso_g"
    if "cephalothorax" in c: rename[c] = "cefalotorax_cm"
raw = raw.rename(columns=rename)
# Convertir a numérico
for c in ("longitud_cm", "peso_g", "cefalotorax_cm"):
    if c in raw.columns:
        raw[c] = pd.to_numeric(raw[c], errors="coerce")
# Filtrar datos válidos
raw = raw[(raw["longitud_cm"] > 0) & (raw["peso_g"] > 0)].reset_index(drop=True)

print("Dimensiones finales:", raw.shape)
print(raw.head())
# Guardar unificado
out = BASE / "maestro_biometria.csv"
raw.to_csv(out, index=False)
print("Guardado en:", out)

- DB1INDUSTRIALFARM1.xlsx
- DB2INDUSTRIALFARM2C1.xlsx
- DB3INDUSTRIALFARM2C2.xlsx
- DB4ACADEMICPONDS1.xlsx
- DB5ACADEMICPONDS2.xlsx
- IoT_AquaSensors_Crudo.csv
- maestro_biometria.csv
- Ponds1.csv
Leyendo: /mnt/c/Data/DB1INDUSTRIALFARM1.xlsx
Leyendo: /mnt/c/Data/DB2INDUSTRIALFARM2C1.xlsx
Leyendo: /mnt/c/Data/DB3INDUSTRIALFARM2C2.xlsx
Leyendo: /mnt/c/Data/DB4ACADEMICPONDS1.xlsx
Leyendo: /mnt/c/Data/DB5ACADEMICPONDS2.xlsx
Dimensiones finales: (170, 6)
   sample  longitud_cm  peso_g  complete_shrimp_images  \
0       1        12.20    33.0                       1   
1       2        14.40    33.0                       1   
2       3        12.24    31.0                       1   
3       4        12.40    34.0                       0   
4       5        11.00    19.0                       0   

               source_file  cefalotorax_cm  
0  DB1INDUSTRIALFARM1.xlsx             NaN  
1  DB1INDUSTRIALFARM1.xlsx             NaN  
2  DB1INDUSTRIALFARM1.xlsx             NaN  
3  DB1INDUSTRIALF